In [2]:
# pip install --upgrade jupyterlab-git

In [34]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
import pandas as pd
import numpy as np
import numpy.random as npr
import pytest
from pandas.testing import assert_frame_equal
from collections import defaultdict

In [42]:
sys.path.append(os.path.abspath("/Users/andresmondragon/nomad/daphme"))

In [48]:
import stop_detection as SD

In [156]:
import pandas as pd
import numpy as np

# Sample data to create a DataFrame
data_dict = {
    'cluster': [1, 1, 2, 2, 1, 1, 3, 3, 1],  # Example clusters
    'x': [1000, 1020, 1150, 1180, 1300, 1255, 1500, 1550, 1000],
    'y': [2000, 2025, 2100, 2150, 2300, 2250, 2500, 2550, 2050],
}

# Create the DataFrame
data = pd.DataFrame(data_dict)

# Set the index to Unix timestamp (in seconds)
data.index = pd.date_range(start='2024-01-01 00:00:00', periods=9, freq='H').astype(np.int64) // 10**9

data

,cluster,x,y
1704067200,1,1000,2000
1704070800,1,1020,2025
1704074400,2,1150,2100
1704078000,2,1180,2150
1704081600,1,1300,2300
1704085200,1,1255,2250
1704088800,3,1500,2500
1704092400,3,1550,2550
1704096000,1,1000,2050


In [158]:
def find_neighbors(data, time_thresh, dist_thresh):
    """
    Identifies neighboring pings for each user ping within specified time and distance thresholds.

    Parameters
    ----------
    df : pandas.DataFrame
        User pings with 'x' (EPSG:3857), 'y' (EPSG:3857) columns, indexed by 'unix_timestamp'
    time_thresh : int
        Time threshold in minutes.
    dist_thresh : float
        Distance threshold in meters.

    Returns
    -------
    dict
        Neighbors indexed by unix timestamp, with values as sets of neighboring unix timestamps.
    """    
    
    unix_timestamps, x, y = data.index.values, data['x'].values, data['y'].values

    # Time threshold calculation using broadcasting
    within_threshold = np.triu(np.abs(unix_timestamps[:, np.newaxis] - unix_timestamps) <= (time_thresh * 60), k=1)
    t_pairs = np.where(within_threshold)

    # Distance calculation
    distances_sq = (x[t_pairs[0]] - x[t_pairs[1]])**2 + (y[t_pairs[0]] - y[t_pairs[1]])**2
    neighbor_pairs = distances_sq < dist_thresh**2

    # Building the neighbor dictionary
    neighbor_dict = defaultdict(set)
    for i, j in zip(t_pairs[0][neighbor_pairs], t_pairs[1][neighbor_pairs]):
        neighbor_dict[unix_timestamps[i].item()].add(unix_timestamps[j].item())
        neighbor_dict[unix_timestamps[j].item()].add(unix_timestamps[i].item())

    return neighbor_dict

In [164]:
find_neighbors(data, 120, 100)

defaultdict(set,
            {1704067200: {1704070800},
             1704070800: {1704067200},
             1704074400: {1704078000},
             1704078000: {1704074400},
             1704081600: {1704085200},
             1704085200: {1704081600},
             1704088800: {1704092400},
             1704092400: {1704088800}})

In [174]:
def dbscan(data, time_thresh, dist_thresh, min_pts, neighbor_dict=None):
    """
    Implements DBSCAN.

    Parameters
    ----------
    data : pandas.DataFrame
        User pings with 'x' (EPSG:3857), 'y' (EPSG:3857) columns, indexed by 'unix_timestamp'
    time_thresh : int
        Time threshold in minutes.
    dist_thresh : float
        Distance threshold in meters.
    min_pts: int
        A cluster must have at least (min_pts+1) points to be considered a cluster.

    Returns
    -------
    pandas.DataFrame
        Contains two columns 'cluster' (int), 'core' (int) labeling each ping with their cluster id and core id or noise, indexed by 'unix_timestamp'
    """    
    if not neighbor_dict:
        neighbor_dict = find_neighbors(data, time_thresh, dist_thresh)
    else:
        valid_times = set(data.index)
        neighbor_dict = defaultdict(set, {k: v.intersection(valid_times) for k, v in neighbor_dict.items() if k in valid_times})
    
    cluster_df = pd.Series(-2, index=data.index, name='cluster')
    core_df = pd.Series(-1, index=data.index, name='core')
    
    cid = -1                                              # Initialize cluster label
    for i, cluster in cluster_df.items():
        if cluster < 0:                                   # Check if point is not yet in a cluster
            if len(neighbor_dict[i]) < min_pts:
                cluster_df[i] = -1                        # Mark as noise if below min_pts
            else:
                cid += 1
                cluster_df[i] = cid                       # Assign new cluster label
                core_df[i] = cid                          # Assign new core label
                S = list(neighbor_dict[i])                # Initialize stack with neighbors
                
                while S:
                    j = S.pop()
                    if cluster_df[j] < 0:                 # Process if not yet in a cluster
                        cluster_df[j] = cid
                        if len(neighbor_dict[j]) >= min_pts:
                            core_df[j] = cid              # Assign core label
                            for k in neighbor_dict[j]:
                                if cluster_df[k] < 0:
                                    S.append(k)           # Add new neighbors
    
    return pd.DataFrame({'cluster': cluster_df, 'core': core_df})

In [287]:
pings_dict = {
        'x': [1000, 1005, 1010, 1200, 1205, 1207, 1400, 1403, 1008, 1012, 1206, 1402],
        'y': [2000, 2002, 2005, 2200, 2202, 2203, 2400, 2401, 2008, 2003, 2204, 2402],
    }
pings = pd.DataFrame(pings_dict)
pings.index = pd.to_datetime(['2024-01-01 00:00:00', '2024-01-01 00:01:00', '2024-01-01 00:02:00',
                              '2024-01-01 01:00:00', '2024-01-01 01:01:00', '2024-01-01 01:02:00',
                              '2024-01-01 02:00:00', '2024-01-01 02:01:00', '2024-01-01 00:03:00',
                              '2024-01-01 00:04:00', '2024-01-01 01:03:00', '2024-01-01 02:02:00'
                             ]).astype(np.int64) // 10**9

pings

,x,y
1704067200,1000,2000
1704067260,1005,2002
1704067320,1010,2005
1704070800,1200,2200
1704070860,1205,2202
1704070920,1207,2203
1704074400,1400,2400
1704074460,1403,2401
1704067380,1008,2008
1704067440,1012,2003


In [289]:
len(pings)

12

In [299]:
time_thresh = 120  # Time threshold in minutes
dist_thresh = 100 # Distance threshold in meters
min_pts = 2

In [293]:
cluster_results = dbscan(pings, time_thresh, dist_thresh, min_pts)
cluster_results

,cluster,core
1704067200,0,0
1704067260,0,0
1704067320,0,0
1704070800,1,1
1704070860,1,1
1704070920,1,1
1704074400,2,2
1704074460,2,2
1704067380,0,0
1704067440,0,0


In [295]:
len(cluster_results)

12

In [303]:
def process_clusters(data, time_thresh, dist_thresh, min_pts, output, cluster_df=None, neighbor_dict=None, min_duration=4):
    if not neighbor_dict:
        neighbor_dict = find_neighbors(data, time_thresh, dist_thresh)
    if cluster_df is None:    # First call of process_clusters
        cluster_df = dbscan(data=data, time_thresh=time_thresh, dist_thresh=dist_thresh, min_pts=min_pts, neighbor_dict=neighbor_dict)
    if len(cluster_df) < min_pts:
        return False
        
    cluster_df = cluster_df[cluster_df['cluster'] != -1]    # Remove noise pings
    
    # All pings are in the same cluster
    if len(cluster_df['cluster'].unique()) == 1:
        x = dbscan(data=data.loc[cluster_df.index], time_thresh=time_thresh, dist_thresh=dist_thresh, min_pts=min_pts, neighbor_dict=neighbor_dict)   # We rerun dbscan because possibly these points no longer hold their own
        y = x.loc[x['cluster'] != -1] 
        z = x.loc[x['core'] != -1]
        
        # There is exactly 1 cluster of all the same values
        if len(y) > 0:
            duration = int((y.index.max() - y.index.min()) // 60)    # Assumes unix_timestamp is in seconds
            if duration > min_duration:
                cid = max(output['cluster']) + 1   # Create new cluster id
                output['cluster'].loc[y.index] = cid
                output['core'].loc[z.index] = cid
            return True
        elif len(y)==0:    # The points in df, despite originally being part of a cluster, no longer hold their own
            return False
        
    # There is more than one cluster
    else:        
        i, j = extract_middle(cluster_df)    # Indices of the "middle" of the cluster (i.e., the head is the first contiguous cluster, and the middle follows that)
        # Recursively processes clusters
        if process_clusters(data, time_thresh, dist_thresh, min_pts, output, cluster_df[i:j]): # Valid cluster in the middle
            process_clusters(data, time_thresh, dist_thresh, min_pts, output, cluster_df[:i])  # Process the initial stub
            process_clusters(data, time_thresh, dist_thresh, min_pts, output, cluster_df[j:])  # Process the "tail"
            return True
        else: # No valid cluster in the middle
            return process_clusters(data, time_thresh, dist_thresh, min_pts, output, pd.concat( [cluster_df[:i],cluster_df[j:]] )) #what if this is out of bounds?

In [305]:
output = pd.DataFrame({'cluster': -1, 'core': -1}, index=data.index)

In [309]:
process_clusters(pings, time_thresh, dist_thresh, 20, output)

False

In [313]:
def medoid(coords):
    """
    Computes the medoid of a set of coordinates. The medoid is defined to be the coordinate 
    in a set that minimizes the maximum distance to every other coordinate in the set.

    Parameters
    ----------
    coords: numpy array
        n x 2 array of pings (x, y).

    Returns
    -------
    numpy array of shape (1,2) denoting medroid coordinates.
    """
    
    # Create matrix of all pairwise distances
    x = coords[:, 0]
    y = coords[:, 1]
    x_diff = x[:, np.newaxis] - x
    y_diff = y[:, np.newaxis] - y
    distances = np.sqrt(x_diff**2 + y_diff**2)
    
    max_distances = np.amax(distances, axis=1)
    medoid_index = np.argmin(max_distances)
    
    return coords[medoid_index,:]

In [315]:
coords = np.array([
    [1, 1],
    [2, 2],
    [3, 3],
    [6, 6]
])

In [317]:
medoid(coords)

array([3, 3])

In [319]:
def diameter(coords):
    """
    Computes the diameter of a set of coordinates, where the diameter is defined to be the greatest 
    distance between any two coordinates in a set.

    Parameters
    ----------
    coords: numpy array
        n x 2 array of pings (x, y).

    Returns
    -------
    float denoting diameter.
    """
    
    # Create matrix of all pairwise distances
    x = coords[:, 0]
    y = coords[:, 1]
    x_diff = x[:, np.newaxis] - x
    y_diff = y[:, np.newaxis] - y
    distances = np.sqrt(x_diff**2 + y_diff**2)
    
    return np.max(distances)

In [323]:
diameter(coords)

7.0710678118654755

In [325]:
import string

In [327]:
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [329]:
string.whitespace

' \t\n\r\x0b\x0c'

In [331]:
np.empty((0,4))

array([], shape=(0, 4), dtype=float64)